In [5]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("khushikyad001/electric-vehicle-analytics-dataset")

# print("Path to dataset files:", path)

In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, PolynomialFeatures
from sklearn.compose import ColumnTransformer

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.optim as optim

import pytorch_lightning as pl
from pytorch_lightning import LightningModule
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from torchmetrics.regression import R2Score, MeanSquaredError, MeanAbsoluteError
from torchmetrics.functional import r2_score, mean_squared_error, mean_absolute_error

import pytorch_optimizer as optim1
import optuna
pd.set_option('display.max_columns', 1000)
pl.seed_everything(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Seed set to 42


In [7]:
df = pd.read_csv('electric_vehicle_analytics.csv', index_col='Vehicle_ID')
df

,Make,Model,Year,Region,Vehicle_Type,Battery_Capacity_kWh,Battery_Health_%,Range_km,Charging_Power_kW,Charging_Time_hr,Charge_Cycles,Energy_Consumption_kWh_per_100km,Mileage_km,Avg_Speed_kmh,Max_Speed_kmh,Acceleration_0_100_kmh_sec,Temperature_C,Usage_Type,CO2_Saved_tons,Maintenance_Cost_USD,Insurance_Cost_USD,Electricity_Cost_USD_per_kWh,Monthly_Charging_Cost_USD,Resale_Value_USD
Vehicle_ID,,,,,,,,,,,,,,,,,,,,,,,,
1,Nissan,Leaf,2021,Asia,SUV,101.7,75.5,565,153.6,0.82,1438,12.76,117727,53.4,233,8.10,-9.0,Personal,14.13,969,843,0.30,375.55,26483
2,Nissan,Leaf,2020,Australia,Sedan,30.1,99.8,157,157.2,0.27,1056,15.79,161730,58.0,221,9.83,1.6,Personal,19.41,1157,1186,0.25,532.02,11287
3,Hyundai,Kona Electric,2021,North America,SUV,118.5,84.0,677,173.6,0.84,1497,24.34,244931,69.4,138,3.60,1.5,Fleet,29.39,291,1890,0.26,1291.68,34023
4,Audi,Q4 e-tron,2022,Europe,Hatchback,33.1,97.3,149,169.3,0.25,1613,14.70,57995,42.9,192,8.97,12.5,Fleet,6.96,401,2481,0.33,234.44,14398
5,Tesla,Model 3,2022,Australia,Truck,81.3,85.6,481,212.8,0.43,1078,22.77,17185,97.6,189,7.03,-3.0,Commercial,2.06,214,2336,0.10,32.61,23033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2996,Mercedes,EQS,2021,North America,SUV,57.2,84.0,239,102.2,0.69,1242,21.39,130400,43.2,154,7.98,33.7,Personal,15.65,1645,2357,0.33,767.05,16749
2997,Ford,Mustang Mach-E,2022,Europe,Hatchback,98.4,83.1,498,160.6,0.74,1793,18.62,15175,41.8,243,7.22,13.7,Fleet,1.82,289,868,0.32,75.35,30080
2998,Kia,Niro EV,2024,Europe,Truck,35.1,82.1,189,18.1,2.56,1184,20.34,219055,33.6,175,4.26,5.8,Fleet,26.29,1013,695,0.32,1188.15,16286


In [ ]:
X_num = df.select_dtypes(exclude='object')
X_cat = df.select_dtypes(include='object')
X_num = X_num.drop(columns='Resale_Value_USD', axis=1)
y = df.Resale_Value_USD

X_num_train, X_num_test, X_cat_train, X_cat_test, y_train, y_test = train_test_split(X_num, X_cat, y, random_state=42, test_size=0.2, )

,Year,Battery_Capacity_kWh,Battery_Health_%,Range_km,Charging_Power_kW,Charging_Time_hr,Charge_Cycles,Energy_Consumption_kWh_per_100km,Mileage_km,Avg_Speed_kmh,Max_Speed_kmh,Acceleration_0_100_kmh_sec,Temperature_C,CO2_Saved_tons,Maintenance_Cost_USD,Insurance_Cost_USD,Electricity_Cost_USD_per_kWh,Monthly_Charging_Cost_USD,Resale_Value_USD
Vehicle_ID,,,,,,,,,,,,,,,,,,,
1,2021,101.7,75.5,565,153.6,0.82,1438,12.76,117727,53.4,233,8.10,-9.0,14.13,969,843,0.30,375.55,26483
2,2020,30.1,99.8,157,157.2,0.27,1056,15.79,161730,58.0,221,9.83,1.6,19.41,1157,1186,0.25,532.02,11287
3,2021,118.5,84.0,677,173.6,0.84,1497,24.34,244931,69.4,138,3.60,1.5,29.39,291,1890,0.26,1291.68,34023
4,2022,33.1,97.3,149,169.3,0.25,1613,14.70,57995,42.9,192,8.97,12.5,6.96,401,2481,0.33,234.44,14398
5,2022,81.3,85.6,481,212.8,0.43,1078,22.77,17185,97.6,189,7.03,-3.0,2.06,214,2336,0.10,32.61,23033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2996,2021,57.2,84.0,239,102.2,0.69,1242,21.39,130400,43.2,154,7.98,33.7,15.65,1645,2357,0.33,767.05,16749
2997,2022,98.4,83.1,498,160.6,0.74,1793,18.62,15175,41.8,243,7.22,13.7,1.82,289,868,0.32,75.35,30080
2998,2024,35.1,82.1,189,18.1,2.56,1184,20.34,219055,33.6,175,4.26,5.8,26.29,1013,695,0.32,1188.15,16286
